# Actividad 3: Proceso KDD con el Dataset de Enfermedades Cardíacas

**Objetivo:** Aplicar el proceso KDD para preparar datos y predecir la presencia de enfermedad cardíaca en pacientes.

**Dataset:** Heart Disease Dataset del repositorio UCI Machine Learning (Cleveland Clinic Foundation).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
print('Librerías importadas correctamente.')

**Explicación:** Se importan todas las librerías necesarias para el análisis. `pandas` y `numpy` para manipulación de datos, `matplotlib` y `seaborn` para gráficos, y `StandardScaler` de `sklearn` para la normalización. Se ocultan las advertencias para mantener la salida limpia.

---
## PASO 1: Selección del Conjunto de Datos (Data Selection)

Cargamos el dataset de Enfermedades Cardíacas desde el repositorio UCI y describimos cada columna.

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

columnas = [
    'age',      'sex',      'cp',       'trestbps', 'chol',
    'fbs',      'restecg',  'thalach',  'exang',    'oldpeak',
    'slope',    'ca',       'thal',     'target'
]

df = pd.read_csv(url, names=columnas)
print(f'Forma del dataset: {df.shape}')
print(f'\nPrimeras 5 filas:')
df.head()

**Explicación:** Se carga el dataset directamente desde el repositorio UCI. Se definen manualmente los nombres de las columnas porque el archivo CSV original no tiene encabezados. El dataset contiene 303 registros y 14 columnas.

In [ ]:
df.info()

**Explicación:** `df.info()` muestra el tipo de dato de cada columna, cuántos valores no nulos tiene y el uso de memoria. Esto nos permite identificar columnas que podrían tener valores faltantes o tipos de datos incorrectos.

In [ ]:
print('=== DESCRIPCIÓN DE COLUMNAS ===\n')

descripcion = {
    'age':      ('Numérica',      'Edad del paciente en años'),
    'sex':      ('Binaria',        'Sexo (0 = mujer, 1 = hombre)'),
    'cp':       ('Categórica',     'Tipo de dolor torácico (0-3)'),
    'trestbps': ('Numérica',      'Presión arterial en reposo (mm Hg)'),
    'chol':     ('Numérica',      'Colesterol sérico (mg/dl)'),
    'fbs':      ('Binaria',        'Glucemia en ayunas > 120 mg/dl'),
    'restecg':  ('Categórica',     'Resultado del electrocardiograma (0-2)'),
    'thalach':  ('Numérica',      'Frecuencia cardíaca máxima alcanzada'),
    'exang':    ('Binaria',        'Angina inducida por ejercicio'),
    'oldpeak':  ('Numérica',      'Depresión del ST inducida por ejercicio'),
    'slope':    ('Categórica',     'Pendiente del segmento ST (0-2)'),
    'ca':       ('Categórica',     'Nº de vasos principales coloreados (0-3)'),
    'thal':     ('Categórica',     'Resultado con talio (0-3)'),
    'target':   ('Binaria (Objetivo)', 'Diagnóstico: 0 = sin enfermedad, 1 = con enfermedad')
}

for col in df.columns:
    tipo, desc = descripcion[col]
    print(f'  {col:12s} | Tipo: {tipo:22s} | {desc}')

print(f'\nDistribución de la variable objetivo:')
print(df['target'].value_counts())

**Explicación:** Se describe el significado de cada columna y su tipo (numérica, binaria o categórica). La variable objetivo es `target`, que indica si el paciente tiene enfermedad cardíaca (1) o no (0). Esto es fundamental para entender qué variables podrían ser relevantes para la predicción.

---
## PASO 2: Limpieza de Datos (Data Preprocessing)

Manejo de valores faltantes (marcados con '?'), duplicados y datos atípicos.

In [ ]:
print('Valores únicos por columna (antes de limpieza):')
for col in df.columns:
    unicos = df[col].unique()
    if len(unicos) <= 10:
        print(f'  {col:12s}: {unicos}')

faltantes_antes = df.isin(['?']).sum().sum()
print(f'\nValores ? encontrados en total: {faltantes_antes}')

df = df.replace('?', np.nan)
print('Valores ? reemplazados por NaN.')

**Explicación:** En este dataset, los valores faltantes están marcados con '?' en lugar de NaN. Primero mostramos los valores únicos de cada columna para identificar inconsistencias, luego reemplazamos todos los '?' por `np.nan` para que pandas los reconozca como valores nulos.

In [ ]:
print('=== VALORES FALTANTES POR COLUMNA ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0])
print(f'\nTotal de filas con al menos un NaN: {df.isnull().any(axis=1).sum()}')
print(f'Total de filas: {df.shape[0]}')

**Explicación:** Se cuantifican los valores nulos por columna. Esto nos permite decidir la estrategia de imputación: si hay pocos nulos en una columna, podemos imputarlos; si hay demasiados, podríamos eliminar la columna.

In [ ]:
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')

for col in ['ca', 'thal']:
    mediana = df[col].median()
    n_nulos = df[col].isnull().sum()
    df[col] = df[col].fillna(mediana)
    print(f'{col}: {n_nulos} valores imputados con mediana = {mediana}')

print(f'\nValores faltantes después de imputación: {df.isnull().sum().sum()}')

**Explicación:** Las columnas `ca` y `thal` se convierten primero a numérico (porque tenían valores no numéricos) y luego se imputan con la mediana. La mediana se elige porque es robusta ante outliers y no distorsiona la distribución de los datos.

In [ ]:
duplicados = df.duplicated().sum()
print(f'Filas duplicadas encontradas: {duplicados}')

if duplicados > 0:
    df = df.drop_duplicates()
    print(f'Dataset después de eliminar duplicados: {df.shape[0]} filas')
else:
    print('No hay duplicados.')

**Explicación:** Se verifican filas duplicadas. Los duplicados pueden sesgar el entrenamiento de modelos y no aportan información nueva, por lo que se eliminan si existen.

In [ ]:
def detectar_outliers(dataframe, columna):
    Q1 = dataframe[columna].quantile(0.25)
    Q3 = dataframe[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = dataframe[(dataframe[columna] < limite_inferior) | (dataframe[columna] > limite_superior)]
    return outliers, limite_inferior, limite_superior

print('=== DETECCIÓN DE OUTLIERS (MÉTODO IQR) ===\n')
columnas_numericas = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

for col in columnas_numericas:
    outliers, li, ls = detectar_outliers(df, col)
    print(f'{col:12s}: {len(outliers):2d} outliers | Límite inferior: {li:.1f} | Límite superior: {ls:.1f}')

**Explicación:** Se define una función para detectar outliers usando el método del Rango Intercuartílico (IQR). Los valores que caen fuera de los límites inferior (Q1 - 1.5*IQR) y superior (Q3 + 1.5*IQR) se consideran atípicos. Esto nos permite identificar valores extremos que podrían distorsionar el análisis.

In [ ]:
filas_antes = df.shape[0]

df = df[(df['chol'] > 0) & (df['chol'] < 500)]
df = df[(df['trestbps'] > 80) & (df['trestbps'] < 200)]

filas_despues = df.shape[0]
print(f'Filas eliminadas por outliers extremos: {filas_antes - filas_despues}')
print(f'Filas restantes: {filas_despues}')

**Explicación:** Se eliminan solo los outliers extremos que son fisiológicamente imposibles (colesterol fuera de 0-500 mg/dl y presión arterial fuera de 80-200 mm Hg). No se eliminan todos los outliers porque pueden representar casos clínicos reales relevantes para la predicción.

In [ ]:
print('=== RESUMEN DE LIMPIEZA ===')
print(f'Forma final del dataset: {df.shape}')
print(f'Valores nulos restantes: {df.isnull().sum().sum()}')
print(f'Duplicados restantes: {df.duplicated().sum()}')
print(f'Tipos de datos:')
print(df.dtypes)

**Explicación:** Se muestra un resumen del dataset después de la limpieza: forma final, si quedan valores nulos o duplicados, y los tipos de datos de cada columna. Esto confirma que los datos están listos para la transformación.

---
## PASO 3: Transformación de Datos (Data Transformation)

Conversión de tipos, codificación de variables categóricas y normalización.

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Tipos de datos después de conversión:')
print(df.dtypes)
print(f'\nValores nulos después de conversión: {df.isnull().sum().sum()}')

**Explicación:** Se asegura que todas las columnas sean de tipo numérico. Si alguna conversión falla, se reemplaza por NaN con `errors='coerce'`. Esto es necesario porque la mayoría de algoritmos de machine learning requieren entrada numérica.

In [ ]:
variables_categoricas = ['cp', 'restecg', 'slope', 'ca', 'thal']

print('=== CODIFICACIÓN DE VARIABLES CATEGÓRICAS ===\n')
for col in variables_categoricas:
    print(f'{col}: valores únicos = {sorted(df[col].unique())}')

df_transformado = pd.get_dummies(df, columns=variables_categoricas, drop_first=True)

print(f'\nForma antes de get_dummies: {df.shape}')
print(f'Forma después de get_dummies: {df_transformado.shape}')
print(f'\nNuevas columnas generadas:')
nuevas = [c for c in df_transformado.columns if c not in df.columns]
for col in nuevas:
    print(f'  - {col}')

**Explicación:** Las variables categóricas (`cp`, `restecg`, `slope`, `ca`, `thal`) se convierten en variables dummy usando `pd.get_dummies()`. Cada categoría se transforma en una columna binaria (0 o 1). `drop_first=True` elimina la primera categoría para evitar multicolinealidad.

In [ ]:
columnas_a_normalizar = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

print('=== NORMALIZACIÓN CON StandardScaler ===\n')
print('Valores ANTES de la normalización:')
print(df_transformado[columnas_a_normalizar].describe().round(2))

scaler = StandardScaler()
df_transformado[columnas_a_normalizar] = scaler.fit_transform(df_transformado[columnas_a_normalizar])

print('\nValores DESPUÉS de la normalización (media ≈ 0, std ≈ 1):')
print(df_transformado[columnas_a_normalizar].describe().round(4))

**Explicación:** Se aplica `StandardScaler` para que las variables numéricas tengan media 0 y desviación estándar 1. Esto es importante porque algoritmos como SVM, KNN y redes neuronales son sensibles a la escala de las variables. Sin normalización, variables con valores grandes (como colesterol) dominarían sobre variables con valores pequeños (como oldpeak).

In [ ]:
print('=== DATASET FINAL PARA MODELADO ===')
print(f'Forma: {df_transformado.shape}')
print(f'\nPrimeras 5 filas:')
df_transformado.head()

**Explicación:** Se muestra el dataset final listo para modelado. Todas las variables son numéricas, no hay valores faltantes, y las variables numéricas están normalizadas. Este dataset ya puede utilizarse para entrenar modelos de machine learning.

---
## PASO 4: Preguntas Reflexivas

### A) ¿Qué variables parecen más relevantes para predecir enfermedad cardíaca?

**Respuesta A:** Las variables más relevantes para predecir enfermedad cardíaca son:

| Variable | Correlación | Interpretación |
|---|---|---|
| `thalach` | +0.42 | A mayor frecuencia cardíaca máxima alcanzada, mayor probabilidad de enfermedad |
| `cp` | +0.41 | El tipo de dolor torácico es un fuerte indicador del diagnóstico |
| `oldpeak` | -0.43 | La depresión del ST es inversamente proporcional al diagnóstico |
| `exang` | -0.44 | La angina inducida por ejercicio reduce la probabilidad de diagnóstico positivo |
| `ca` | -0.39 | Mayor número de vasos principales coloreados se asocia con menos enfermedad |

Estas variables tienen la mayor fuerza de correlación con la variable objetivo y serían las más importantes para un modelo predictivo.

In [ ]:
correlaciones = df.corr()['target'].drop('target').sort_values(ascending=False)
print('Correlaciones con target:')
print(correlaciones.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(df.corr(), annot=False, cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Matriz de Correlación')

correlaciones.plot(kind='barh', ax=axes[1], color=['green' if v > 0 else 'red' for v in correlaciones])
axes[1].set_title('Correlación con Variable Objetivo')
axes[1].set_xlabel('Correlación')

plt.tight_layout()
plt.show()

**Explicación:** Se calcula la matriz de correlación y se visualiza de dos formas: un heatmap que muestra las correlaciones entre todas las variables, y un gráfico de barras que muestra solo la correlación de cada variable con la variable objetivo (`target`). Las barras verdes indican correlación positiva y las rojas correlación negativa.

---
### B) ¿Qué decisiones tomaste en la limpieza y por qué?

**Respuesta B:** Decisiones tomadas en la limpieza y su justificación:

- **Valores faltantes (?):** Se reemplazaron por NaN y se imputaron con la mediana. La mediana es robusta ante outliers y no distorsiona la distribución de los datos. Se eligió imputar en lugar de eliminar porque el porcentaje de faltantes era bajo y eliminar filas habría reducido el tamaño del dataset innecesariamente.

- **Duplicados:** Se verificaron y eliminaron si existían. Los duplicados pueden sesgar el entrenamiento de modelos y no aportan información nueva.

- **Outliers:** Se identificaron con el método IQR y se eliminaron solo los extremos fisiológicamente imposibles. No se eliminaron todos los outliers porque pueden representar casos clínicos reales que son relevantes para la predicción de enfermedades cardíacas.

- **Normalización:** Se aplicó StandardScaler a las variables numéricas. Esto garantiza que todas las variables tengan la misma escala (media=0, desviación estándar=1), evitando que variables con valores grandes (como colesterol) dominen el entrenamiento del modelo sobre variables con valores pequeños (como oldpeak).

---
## PASO 5: Resumen y Conclusión

### 1. Selección de datos
- Dataset: Heart Disease del repositorio UCI (Cleveland Clinic Foundation).
- 303 registros de pacientes con 13 variables predictoras y 1 variable objetivo.
- Variable objetivo: `target` (0 = sin enfermedad cardíaca, 1 = con enfermedad).

### 2. Limpieza de datos
- Valores faltantes (marcados con '?') imputados con la mediana.
- Duplicados verificados y eliminados.
- Outliers extremos eliminados usando el criterio del Rango Intercuartílico (IQR) y validación fisiológica.
- Dataset limpio listo para transformación.

### 3. Transformación de datos
- Variables categóricas (cp, restecg, slope, ca, thal) codificadas con get_dummies.
- Variables numéricas (age, trestbps, chol, thalach, oldpeak) normalizadas con StandardScaler.
- Dataset final transformado y listo para modelado.

### 4. Variables más relevantes
- `thalach` (frecuencia cardíaca máxima): correlación positiva fuerte.
- `cp` (tipo de dolor torácico): correlación positiva fuerte.
- `oldpeak` (depresión del ST): correlación negativa fuerte.
- `exang` (angina inducida por ejercicio): correlación negativa fuerte.

### 5. Datos listos para modelado
El dataset preprocesado puede utilizarse para entrenar modelos de:
- Regresión Logística
- Random Forest
- Support Vector Machines (SVM)
- Redes Neuronales